# Phân Tích Chuyên Sâu: CBAM và Bài Toán Suy Giảm Đặc Trưng Ở Đối Tượng Nhỏ

**Mục tiêu:** Chứng minh một cách định lượng và định tính rằng mô hình Baseline (ResNet50) bị mất thông tin các đối tượng nhỏ do cơ chế Global Average Pooling (cào bằng đặc trưng), trong khi mô hình đề xuất (EfficientNet + CBAM) với cơ chế Attention đã tập trung chính xác vào đối tượng nhỏ và chống lại nhiễu nền (Background Clutter).

In [ ]:
!pip install timm pycocotools torchmetrics scikit-learn matplotlib seaborn pyyaml tqdm -q
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, json
from pathlib import Path
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Setup Workspace
REPO_URL = 'https://github.com/Thinh59/ECAAL.git'
REPO_DIR = Path('/kaggle/working/ECAAL')
if REPO_DIR.exists():
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))

In [ ]:
import shutil
OUTPUTS_DIR = Path('/kaggle/working/outputs')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT = Path('/kaggle/input/datasets/thinhha59/models')
RESULTS_DIR  = DATASET_ROOT / 'results'
SRC_SUBSET   = RESULTS_DIR / 'data' / 'coco_subset'
SUBSET_DIR = Path('/kaggle/working/data/coco_subset')
SUBSET_DIR.mkdir(parents=True, exist_ok=True)
if RESULTS_DIR.exists():
    for fname in ['subset_train_ids.json', 'subset_val_ids.json', 'subset_test_ids.json']:
        if (SRC_SUBSET / fname).exists(): shutil.copy2(SRC_SUBSET / fname, SUBSET_DIR / fname)
    for exp in ['exp_B_resnet_asl', 'exp_C_efficientnet_cbam_asl']:
        src = RESULTS_DIR / exp
        dst = OUTPUTS_DIR / exp
        dst.mkdir(parents=True, exist_ok=True)
        for fname in ['best.pth', 'log.json']:
            if (src / fname).exists(): shutil.copy2(src / fname, dst / fname)

In [ ]:
coco_root = '/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017'
if not os.path.exists(coco_root): print('Vui lòng thêm dữ liệu COCO 2017 (awsaf49/coco-2017-dataset)')

## 1. Load Data và Models

In [ ]:
from dataset import build_dataloaders
cfg = {'data': {'root': coco_root, 'subset_dir': str(SUBSET_DIR)}, 'batch_size': 32, 'num_workers': 2}
_, _, test_loader = build_dataloaders(cfg)

In [ ]:
from models import build_model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_B = build_model({'backbone': 'resnet50', 'use_cbam': False, 'num_classes': 80}).to(device)
model_C = build_model({'backbone': 'efficientnet_b0', 'use_cbam': True, 'num_classes': 80}).to(device)

ckpt_B = OUTPUTS_DIR / 'exp_B_resnet_asl' / 'best.pth'
ckpt_C = OUTPUTS_DIR / 'exp_C_efficientnet_cbam_asl' / 'best.pth'

if ckpt_B.exists(): model_B.load_state_dict(torch.load(ckpt_B, map_location=device, weights_only=True)['model_state_dict'])
if ckpt_C.exists(): model_C.load_state_dict(torch.load(ckpt_C, map_location=device, weights_only=True)['model_state_dict'])

model_B.eval()
model_C.eval()
print("Đã tải 2 mô hình thành công.")

In [ ]:
from tqdm.auto import tqdm
def run_inference(loader):
    all_targets, all_preds_B, all_preds_C, all_ids = [], [], [], []
    with torch.no_grad():
        for images, targets, img_ids in tqdm(loader, desc="Đang đánh giá Test Set"):
            images = images.to(device)
            out_B = torch.sigmoid(model_B(images)).cpu()
            out_C = torch.sigmoid(model_C(images)).cpu()
            all_targets.append(targets.cpu())
            all_preds_B.append(out_B)
            all_preds_C.append(out_C)
            all_ids.extend(img_ids)
    return {'targets': torch.cat(all_targets).numpy(), 'preds_B': torch.cat(all_preds_B).numpy(), 'preds_C': torch.cat(all_preds_C).numpy(), 'ids': np.array(all_ids)}

test_results = run_inference(test_loader)

## 2. Phân Tích Định Lượng Trên Các Lớp Đối Tượng Nhỏ (Small Objects)

Chúng ta sẽ chọn ra một nhóm các đối tượng thường có kích thước rất nhỏ và dễ bị nhiễu nền lấn át trong ảnh COCO. Sau đó, so sánh **False Negative (Tỷ lệ bỏ sót / không phát hiện được)** giữa ResNet và EfficientNet+CBAM.

In [ ]:
# Load thông tin class COCO
with open(coco_root + '/annotations/instances_val2017.json') as f:
    coco_anns = json.load(f)
cats_sorted = sorted(coco_anns['categories'], key=lambda c: c['id'])
coco_classes = [c['name'] for c in cats_sorted]

# Định nghĩa danh sách các đối tượng thường rất nhỏ và khó nhận diện
small_object_names = [
    'spoon', 'sports ball', 'mouse', 'toothbrush', 
    'kite', 'baseball glove', 'cup', 'fork', 'scissors', 'apple', 'carrot'
]
small_obj_indices = [coco_classes.index(name) for name in small_object_names]

threshold = 0.5
targets = test_results['targets'].astype(int)
preds_B = (test_results['preds_B'] > threshold).astype(int)
preds_C = (test_results['preds_C'] > threshold).astype(int)

# Đếm False Negative (Có đối tượng nhưng mô hình đoán là 0)
fn_B = ((targets == 1) & (preds_B == 0)).sum(axis=0)
fn_C = ((targets == 1) & (preds_C == 0)).sum(axis=0)

# Trực quan hoá
fn_B_small = [fn_B[i] for i in small_obj_indices]
fn_C_small = [fn_C[i] for i in small_obj_indices]

x = np.arange(len(small_object_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
rects1 = ax.bar(x - width/2, fn_B_small, width, label='ResNet50 (Baseline)', color='lightcoral')
rects2 = ax.bar(x + width/2, fn_C_small, width, label='EfficientNet+CBAM (Proposed)', color='mediumseagreen')

ax.set_ylabel('Số lượng ảnh bị BỎ SÓT (False Negatives)')
ax.set_title('So sánh Lỗi bỏ sót Đối Tượng Nhỏ: ResNet vs EfficientNet+CBAM')
ax.set_xticks(x)
ax.set_xticklabels(small_object_names, rotation=45)
ax.legend()

def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(rects1)
autolabel(rects2)
fig.tight_layout()
plt.show()

total_fn_B = sum(fn_B_small)
total_fn_C = sum(fn_C_small)
print(f"\nTổng số lỗi bỏ sót (False Negatives) trên nhóm đối tượng nhỏ:")
print(f"- Baseline (ResNet50): {total_fn_B}")
print(f"- Proposed (EfficientNet+CBAM): {total_fn_C}")
print(f"=> CBAM đã giúp giảm {(total_fn_B - total_fn_C)/total_fn_B*100:.2f}% số lần bỏ sót các vật thể nhỏ!")
print("Điều này chứng minh bằng số liệu rằng: GAP cào bằng đặc trưng làm trôi mất thông tin đối tượng nhỏ ở ResNet, trong khi Attention của CBAM giữ lại và khuếch đại chúng.")

## 3. Phân Tích Định Tính (Truy Vết Ảnh Cụ Thể)
Lấy ra danh sách các `Image ID` chứa đối tượng nhỏ (như `spoon` hoặc `kite`) mà ResNet bị mù (Miss), nhưng mô hình của bạn (CBAM) lại nhận diện chính xác. Bạn có thể chép Image ID này vào phần Demo để vẽ Heatmap hoặc giải thích trong báo cáo.

In [ ]:
def find_improvement_cases(class_name):
    c_idx = coco_classes.index(class_name)
    # Tìm ảnh có chứa class này (target == 1)
    # ResNet đoán sai (preds_B == 0) và CBAM đoán đúng (preds_C == 1)
    idx_improved = np.where((targets[:, c_idx] == 1) & (preds_B[:, c_idx] == 0) & (preds_C[:, c_idx] == 1))[0]
    
    if len(idx_improved) > 0:
        print(f"\n[Class: {class_name.upper()}] Tìm thấy {len(idx_improved)} ảnh CBAM cứu được ResNet:")
        # In ra tối đa 5 ID
        for i in range(min(5, len(idx_improved))):
            img_id = test_results['ids'][idx_improved[i]]
            print(f"  - Image ID: {img_id} (ResNet Prob: {test_results['preds_B'][idx_improved[i], c_idx]:.2f} | CBAM Prob: {test_results['preds_C'][idx_improved[i], c_idx]:.2f})")
    else:
        print(f"\n[Class: {class_name.upper()}] Không tìm thấy ca nào khác biệt rõ rệt.")

for cls in ['spoon', 'kite', 'sports ball']:
    find_improvement_cases(cls)